# 02 — Logistic Regression baseline
**Phase 1 deliverable** — a leakage-free Logistic Regression baseline reported on **PR-AUC**, evaluated under both split protocols:

* **Stratified random split** — the convention in most of the literature.
* **Chronological split** — train on the past, test on the future (the realistic protocol; sets up the Phase 4 drift analysis).

We also compare `class_weight=None` vs `class_weight="balanced"` — a free, model-level imbalance strategy that previews the Phase 3 experiment. All later models (RF, XGBoost, DNN) must beat these numbers.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score

from fraud import config, data, evaluation, pipelines

df = data.clean(data.load_raw())
print(f"Working dataset: {df.shape}, frauds: {int(df['Class'].sum())}")

## 1. Train/evaluate under both split protocols

In [ ]:
splits = {
    "stratified": data.stratified_split(df),
    "chronological": data.chronological_split(df),
}
weights = {"unweighted": None, "balanced": "balanced"}

results = {}
for split_name, (X_train, X_test, y_train, y_test) in splits.items():
    for weight_name, cw in weights.items():
        pipe = pipelines.make_logreg_pipeline(class_weight=cw)
        pipe.fit(X_train, y_train)
        metrics = evaluation.evaluate(pipe, X_test, y_test)
        results[(split_name, weight_name)] = (pipe, metrics)
        evaluation.log_result(
            metrics,
            model_name="logistic_regression",
            split=split_name,
            imbalance_strategy=f"class_weight={weight_name}",
            notes="phase1 baseline",
        )
        print(f"=== {split_name} split | class_weight={weight_name} ===")
        evaluation.report(metrics)
        print()

## 2. Side-by-side comparison

In [ ]:
table = pd.DataFrame(
    {name: m for name, (_, m) in results.items()}
).T[["precision", "recall", "f1", "pr_auc", "mcc", "roc_auc"]]
table.index.names = ["split", "class_weight"]
table.round(4)

Reading the table:

* **PR-AUC is the headline number** — it summarises the precision/recall trade-off across all thresholds and is threshold-independent.
* `class_weight="balanced"` typically trades precision for a large recall gain at the default 0.5 threshold; PR-AUC shows whether the *ranking* actually improved.
* Expect the **chronological** split to score somewhat lower than the stratified one — the model is tested on a future time period whose distribution has shifted. That gap is the first concrete evidence of concept drift, quantified properly in Phase 4.


## 3. PR curves and confusion matrices

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for (split_name, weight_name), (pipe, _) in results.items():
    _, X_test, _, y_test = splits[split_name]
    evaluation.plot_pr_curve(pipe, X_test, y_test,
                             label=f"{split_name} / {weight_name}", ax=ax)
ax.set_title("Logistic Regression baseline — PR curves")
fig.savefig(config.FIGURES_DIR / "baseline_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for (split_name, weight_name), (pipe, _) in results.items():
    _, X_test, _, y_test = splits[split_name]
    evaluation.plot_confusion(
        pipe, X_test, y_test,
        title=f"LR — {split_name} / class_weight={weight_name}",
        save_as=f"baseline_cm_{split_name}_{weight_name}.png",
    )
plt.show()

## 4. Cross-validated PR-AUC (the headline baseline number)

In [ ]:
X, y = data.split_features_target(df)
cv = StratifiedKFold(n_splits=config.N_CV_FOLDS, shuffle=True,
                     random_state=config.RANDOM_SEED)

for weight_name, cw in weights.items():
    pipe = pipelines.make_logreg_pipeline(class_weight=cw)
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="average_precision", n_jobs=-1)
    print(f"class_weight={weight_name:>10}: PR-AUC = "
          f"{scores.mean():.4f} +/- {scores.std():.4f}  ({config.N_CV_FOLDS}-fold CV)")

## 5. Conclusion

The 5-fold cross-validated PR-AUC above is the **baseline that every Phase 2 model (Random Forest, XGBoost, DNN with focal loss) must beat**. All metrics were also appended to `results/metrics.csv` with experiment metadata, so the final dissertation tables can be regenerated at any time.

Method notes for the write-up:
* Scaling is fit inside the pipeline on training data only — no leakage.
* Duplicates were removed before splitting so identical rows cannot appear in both train and test.
* The chronological protocol is stricter and more realistic; both are reported to connect with (and critique) the literature.
